# Nyaya — train a small reranker and a bi-encoder on the project's own pairs

The hand-written synonym table gained +0.9 points where it had not been tuned. This
learns the lay-question → statute-section mapping instead, from 4,712 citizen questions
paired with their gold sections and 20 BM25 hard negatives each
(`jitendrajha98/nyaya-retriever-pairs`, built by `scripts/41`).

Two models, both multilingual, both small enough for CPU serving:
- cross-encoder `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` (~118M) — the shippable reranker;
- bi-encoder `intfloat/multilingual-e5-base` fine-tuned with MultipleNegativesRankingLoss.

Evaluation is `scripts/15_retrieval_recall.py` on the never-audited slice, the only
retrieval number this project quotes. Models are saved to the output for upload.

**Settings:** GPU T4 x2, Internet On, Inputs: `jitendrajha98/nyaya-model-src`,
`jitendrajha98/nyaya-retriever-pairs`. ~2-3 GPU-hours.


In [ ]:
import glob, json, os, random, shutil, subprocess, sys, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
PROGRESS = "/kaggle/working/progress.txt"


def note(msg):
    print(msg, flush=True)
    with open(PROGRESS, "a", encoding="utf-8") as fh:
        fh.write(time.strftime("%H:%M:%S ") + msg + chr(10))


def run(cmd):
    note("$ " + " ".join(cmd))
    proc = subprocess.run(cmd, text=True)
    if proc.returncode != 0:
        note(f"FAILED exit {proc.returncode}")
        raise RuntimeError(cmd)
    note("ok")


def find_input(slug, pattern):
    hits = glob.glob(f"/kaggle/input/{slug}/**/{pattern}", recursive=True)
    if not hits:
        import kagglehub
        root = kagglehub.dataset_download(f"jitendrajha98/{slug}")
        hits = glob.glob(os.path.join(root, "**", pattern), recursive=True)
    assert hits, f"{slug}: {pattern} not found"
    return hits[0]


SRC = os.path.dirname(find_input("nyaya-model-src", "pyproject.toml"))
PAIRS = find_input("nyaya-retriever-pairs", "retriever_pairs.jsonl")
WORK = "/kaggle/working/nyaya-model"
shutil.copytree(SRC, WORK, dirs_exist_ok=True)
os.makedirs(f"{WORK}/data/generated", exist_ok=True)
shutil.copy(PAIRS, f"{WORK}/data/generated/retriever_pairs.jsonl")
os.chdir(WORK)
sys.path.insert(0, "src")
run([sys.executable, "-m", "pip", "-q", "install", "-r", "requirements-train.txt"])
note(f"setup ok: pairs={PAIRS}")


In [ ]:
# --- data: pairs -> (query, passage, label) for the cross-encoder; (query, positive) for the bi-encoder
from nyaya.retrieval import load_statute_index
from nyaya.rerank import passage_text

index = load_statute_index("data/canonical")
rows = {f"{r['act_id']}:{r['section'].upper()}": r for r in index.rows}
pairs = [json.loads(l) for l in open("data/generated/retriever_pairs.jsonl", encoding="utf-8")]
random.Random(0).shuffle(pairs)
n_val = 300
val_pairs, train_pairs = pairs[:n_val], pairs[n_val:]
note(f"pairs: train {len(train_pairs)}, val {len(val_pairs)}")


def passage(key):
    return passage_text(rows[key])


ce_train = []
for p in train_pairs:
    rng = random.Random(p["id"])
    for k in p["positive_keys"]:
        ce_train.append((p["query"], passage(k), 1.0))
    for k in rng.sample(p["negative_keys"], min(4, len(p["negative_keys"]))):
        ce_train.append((p["query"], passage(k), 0.0))
random.Random(1).shuffle(ce_train)
note(f"cross-encoder examples: {len(ce_train)}")


In [ ]:
# --- cross-encoder (the shippable reranker) ---------------------------------
import torch
from sentence_transformers import CrossEncoder, InputExample
from torch.utils.data import DataLoader

ce = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1", num_labels=1, max_length=512, device="cuda")
loader = DataLoader([InputExample(texts=[q, p], label=l) for q, p, l in ce_train], shuffle=True, batch_size=32)
t0 = time.time()
ce.fit(train_dataloader=loader, epochs=1, warmup_steps=200, show_progress_bar=False, use_amp=True)
ce.save("/kaggle/working/models/nyaya-reranker-mini-v1")
note(f"cross-encoder trained in {time.time()-t0:.0f}s -> models/nyaya-reranker-mini-v1")


In [ ]:
# --- bi-encoder (first-stage dense retrieval, multilingual) --------------------
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments

bi_rows = [{"anchor": "query: " + p["query"], "positive": "passage: " + passage(k)}
           for p in train_pairs for k in p["positive_keys"]]
bi = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
args = SentenceTransformerTrainingArguments(
    output_dir="/kaggle/working/bi-tmp", num_train_epochs=1, per_device_train_batch_size=32,
    learning_rate=2e-5, warmup_ratio=0.05, fp16=True, logging_steps=50, save_strategy="no", report_to="none")
trainer = SentenceTransformerTrainer(model=bi, args=args, train_dataset=Dataset.from_list(bi_rows),
                                     loss=losses.MultipleNegativesRankingLoss(bi))
t0 = time.time()
trainer.train()
bi.save("/kaggle/working/models/nyaya-embed-v1")
note(f"bi-encoder trained on {len(bi_rows)} pairs in {time.time()-t0:.0f}s -> models/nyaya-embed-v1")
del trainer
torch.cuda.empty_cache()


In [ ]:
# --- evaluate on the never-audited slice (the only number we quote) ------------
run([sys.executable, "scripts/15_retrieval_recall.py", "--k", "1", "3", "5", "8", "--skip-phrase-coverage",
     "--rerank", "/kaggle/working/models/nyaya-reranker-mini-v1", "--rerank-depth", "20", "--max-minutes", "40"])
shutil.copy("reports/retrieval_recall_rerank.json", "/kaggle/working/retrieval_recall_rerank_mini.json")
run([sys.executable, "scripts/15_retrieval_recall.py", "--k", "1", "3", "5", "8", "--skip-phrase-coverage",
     "--dense", "/kaggle/working/models/nyaya-embed-v1", "--max-minutes", "40"])
shutil.copy("reports/retrieval_recall_dense.json", "/kaggle/working/retrieval_recall_dense_embed_v1.json")
for f in ("retrieval_recall_rerank_mini.json", "retrieval_recall_dense_embed_v1.json"):
    r = json.load(open(f"/kaggle/working/{f}"))
    note(f + ": " + json.dumps({k: v.get("full_hit_never_audited", v.get("full_hit")) for k, v in r["recall"].items()}))


In [ ]:
# --- CPU latency of the mini reranker at depth 20 (decides whether it ships) ------
from nyaya.rerank import CrossEncoderReranker

rr = CrossEncoderReranker(model_name="/kaggle/working/models/nyaya-reranker-mini-v1", depth=20, device="cpu")
cands = index.retrieve("police FIR nahi likh rahi, kya karu?", k=20)
rr.rerank("warm up", cands, 5)
t0 = time.time()
rr.rerank("police FIR nahi likh rahi, kya karu?", cands, 5)
note(f"mini reranker CPU latency at depth 20: {time.time() - t0:.2f}s")
shutil.make_archive("/kaggle/working/nyaya-retriever-models", "zip", "/kaggle/working/models")
note("models zipped -> nyaya-retriever-models.zip")
